# IR System — Fix MSMARCO Subset
**Step 12:** Rebuild the MSMARCO subset so it includes **all judged documents (qrels)** + random fill up to 500K, then rebuild everything: index, TF-IDF, embeddings, retrieval results, clustering, and LTR.

⚠️ **Use GPU runtime** (Runtime → Change runtime type → T4 GPU), then Runtime → Run all.

Takes roughly 1.5–2 hours. All outputs are saved to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/ir_system_data'
import os, sys

if not os.path.exists('/content/ir-system'):
    !git clone https://github.com/ghazal-mohammad/ir-system.git /content/ir-system
else:
    !cd /content/ir-system && git pull
sys.path.insert(0, '/content/ir-system')

!pip install ir-datasets==0.5.9 sentence-transformers==2.7.0 scikit-learn nltk -q
print('ready')

In [ ]:
# 1) Build the subset: ALL judged docs + fill with first docs up to 500K
import ir_datasets, json, time
from tqdm import tqdm

TARGET_SIZE = 500_000
dataset = ir_datasets.load('msmarco-passage/trec-dl-2019')

judged_ids = set()
for qrel in dataset.qrels_iter():
    judged_ids.add(qrel.doc_id)
print(f'Judged docs (must be included): {len(judged_ids):,}')

raw_subset = {}
for doc in tqdm(dataset.docs_iter(), total=8841823, desc='Scanning corpus'):
    if doc.doc_id in judged_ids:
        raw_subset[doc.doc_id] = doc.text
    elif len(raw_subset) < TARGET_SIZE - len(judged_ids):
        raw_subset[doc.doc_id] = doc.text
    if len(raw_subset) >= TARGET_SIZE and judged_ids.issubset(raw_subset.keys()):
        break

missing = judged_ids - set(raw_subset.keys())
print(f'Subset size: {len(raw_subset):,} | judged docs missing: {len(missing)}')
assert len(missing) == 0, 'Some judged docs were not found!'

In [ ]:
# 2) Preprocess the subset and save (same pipeline as before)
from services.preprocessing_service import preprocess_to_string

processed_docs = {}
for doc_id, text in tqdm(raw_subset.items(), desc='Preprocessing'):
    processed_docs[doc_id] = preprocess_to_string(text)

with open(f'{SAVE_DIR}/msmarco_docs_processed.json', 'w') as f:
    json.dump(processed_docs, f)
print(f'Saved msmarco_docs_processed.json ({len(processed_docs):,} docs)')

In [ ]:
# 3) Rebuild inverted index + doc lengths + BM25 params
from services.indexing_service import (
    build_inverted_index, get_doc_lengths, get_avg_doc_length, save_index)
from services.bm25_service import save_bm25_params

index = build_inverted_index(processed_docs)
doc_lengths = get_doc_lengths(processed_docs)
avg_dl = get_avg_doc_length(doc_lengths)

save_index(index, f'{SAVE_DIR}/msmarco_index.pkl')
with open(f'{SAVE_DIR}/msmarco_doc_lengths.json', 'w') as f:
    json.dump(doc_lengths, f)
save_bm25_params(avg_dl, f'{SAVE_DIR}/msmarco_bm25_params.pkl')
print(f'Index rebuilt: vocab={len(index):,}, avg_dl={avg_dl:.1f}')

In [ ]:
# 4) Rebuild TF-IDF matrix
from services.tfidf_service import build_tfidf_matrix, save_tfidf_matrix

vectorizer, doc_matrix, tfidf_doc_ids = build_tfidf_matrix(processed_docs)
save_tfidf_matrix(vectorizer, doc_matrix, tfidf_doc_ids, f'{SAVE_DIR}/msmarco')

In [ ]:
# 5) Rebuild embeddings (GPU — ~40 min for 500K)
from services.embedding_service import load_model, encode_texts, save_embeddings

model = load_model()
doc_ids = list(raw_subset.keys())
doc_texts = [raw_subset[d] for d in doc_ids]

embeddings = encode_texts(doc_texts, model, batch_size=256)
save_embeddings(doc_ids, embeddings, f'{SAVE_DIR}/msmarco')
print('Embeddings rebuilt:', embeddings.shape)

In [ ]:
# 6) Re-run retrieval for all queries with all models → save results
from services.preprocessing_service import preprocess
from services.bm25_service import retrieve_bm25
from services.tfidf_service import retrieve_tfidf_fast
from services.embedding_service import retrieve_embedding
from services.hybrid_service import hybrid_parallel, hybrid_serial

raw_queries = {q.query_id: q.text for q in dataset.queries_iter()}
print(f'Queries: {len(raw_queries)}')

bm25_results, tfidf_results, emb_results = {}, {}, {}
hp_results, hs_results = {}, {}

for qid, q_text in tqdm(raw_queries.items(), desc='Retrieval'):
    tokens = preprocess(q_text)
    pq = ' '.join(tokens)
    b_res = retrieve_bm25(tokens, index, doc_lengths, avg_dl, top_k=1000)
    t_res = retrieve_tfidf_fast(pq, vectorizer, doc_matrix, tfidf_doc_ids, top_k=1000)
    e_res = retrieve_embedding(q_text, model, doc_ids, embeddings, top_k=1000)
    bm25_results[qid], tfidf_results[qid], emb_results[qid] = b_res, t_res, e_res
    hp_results[qid] = hybrid_parallel(b_res, t_res, e_res, top_k=1000)
    hs_results[qid] = hybrid_serial(q_text, b_res, model, doc_ids, embeddings,
                                    first_stage_k=100, top_k=1000)

for name, res in [('bm25', bm25_results), ('tfidf', tfidf_results),
                  ('embedding', emb_results), ('hybrid_parallel', hp_results),
                  ('hybrid_serial', hs_results)]:
    with open(f'{SAVE_DIR}/msmarco_{name}_results.json', 'w') as f:
        json.dump(res, f)
print('All retrieval results saved')

In [ ]:
# 7) Re-cluster MSMARCO subset
from services.clustering_service import (
    find_optimal_k, cluster_documents, get_cluster_summary, save_clustering)

opt = find_optimal_k(embeddings, k_range=range(5, 16), sample_size=20000)
print(f'Best K: {opt["best_k"]}')

labels, km = cluster_documents(embeddings, n_clusters=opt['best_k'])
save_clustering(labels, km, f'{SAVE_DIR}/msmarco')

summary = get_cluster_summary(doc_ids, labels, index, opt['best_k'])
with open(f'{SAVE_DIR}/msmarco_cluster_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
for cid, info in summary.items():
    print(f'Cluster {cid}: {info["size"]:,} docs | {info["top_terms"][:5]}')

In [ ]:
# 8) Retrain LTR on the fixed subset (official binarization: relevance >= 2)
from services.ltr_service import (
    build_feature_matrix_multi, train_ltr_model,
    rerank_with_ltr, evaluate_ltr, save_ltr_model)

qrels = {}
for qrel in dataset.qrels_iter():
    if qrel.relevance >= 2:
        qrels.setdefault(qrel.query_id, {})[qrel.doc_id] = qrel.relevance
print(f'Queries with qrels (rel>=2): {len(qrels)}')

results_per_model = {'bm25': bm25_results, 'tfidf': tfidf_results,
                     'embedding': emb_results}
query_ids = list(qrels.keys())
split = int(len(query_ids) * 0.7)
train_qids, test_qids = query_ids[:split], query_ids[split:]
model_names = list(results_per_model.keys())

X_train, y_train, _ = build_feature_matrix_multi(train_qids, results_per_model, qrels)
X_test, y_test, _ = build_feature_matrix_multi(test_qids, results_per_model, qrels)
ltr = train_ltr_model(X_train, y_train)
print('Train:', evaluate_ltr(X_train, y_train, ltr))
print('Test: ', evaluate_ltr(X_test, y_test, ltr))
save_ltr_model(ltr, f'{SAVE_DIR}/msmarco_ltr_model.pkl')

ltr_results = {qid: rerank_with_ltr(qid, results_per_model, ltr, model_names,
                                    top_k=1000) for qid in test_qids}
with open(f'{SAVE_DIR}/msmarco_ltr_results.json', 'w') as f:
    json.dump(ltr_results, f)
print('LTR retrained and saved')

In [ ]:
# 9) Final evaluation — all models on the fixed subset (rel >= 2)
from services.evaluation_service import evaluate_run, print_results_table

all_evals = {}
for name, res in [('TF-IDF', tfidf_results), ('BM25', bm25_results),
                  ('Embedding', emb_results), ('Hybrid Parallel', hp_results),
                  ('Hybrid Serial', hs_results)]:
    runs = {qid: res[qid] for qid in qrels if qid in res}
    all_evals[name] = evaluate_run(runs, qrels)['aggregated']

ltr_eval = evaluate_run(ltr_results, qrels)['aggregated']
all_evals['LTR (test set)'] = ltr_eval

print('=== MSMARCO (fixed subset, rel>=2) ===')
print_results_table(all_evals)

with open(f'{SAVE_DIR}/msmarco_eval_results.json', 'w') as f:
    json.dump(all_evals, f, indent=2)
print('\nSaved msmarco_eval_results.json — take a screenshot of this table!')